# LorentzFM Baseline — Complete Training & Evaluation Notebook

**CareerCompass AI — Research Baseline Comparison**  
**Model:** LorentzFM (Lorentz / Hyperbolic Factorization Machine)  
**Task:** Resume-to-Job Recommendation  
**Metrics:** Precision@10, MRR, NDCG@10, Hit@10

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Configuration & Paths |
| 3 | Feature Engineering — Resume & Job Matrices |
| 4 | LorentzFM Model (Lorentz Manifold + BPR Loss) |
| 5 | Training Dataset & DataLoader |
| 6 | Training Loop with Early Stopping |
| 7 | Training Curves |
| 8 | Evaluation — Ranking Metrics on Full Candidate Pool |
| 9 | Results & Comparison with SBERT |

---

> **Note:** This notebook is self-contained. All code from `dataset.py`, `lorentz_fm.py`, `train.py`, and `evaluate.py` is inlined here for standalone execution.

---
## Section 1 — Setup & Imports

In [ ]:
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

---
## Section 2 — Configuration & Paths

In [ ]:
# ── Root paths ─────────────────────────────────────────────────────────────────
# Adjust WORKSPACE_ROOT if running from a different location
WORKSPACE_ROOT  = Path(r'C:/Users/MRUDANG WANJARI/Desktop/LorrentzFM')
EXPERIMENT_DIR  = WORKSPACE_ROOT / 'lorentzfm_experiment'
RESULTS_DIR     = EXPERIMENT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset paths ──────────────────────────────────────────────────────────────
BERT_DATA_DIR   = Path(r'C:/Users/MRUDANG WANJARI/Desktop/BERT/bert_experiment/data')

RESUMES_PATH        = BERT_DATA_DIR / 'resumes_parsed.csv'
RESUME_DOMAIN_PATH  = BERT_DATA_DIR / 'resume_domain_labels.csv'
DOMAIN_LABELS_PATH  = BERT_DATA_DIR / 'domain_labels_wide.csv'
SPLIT_MANIFEST_PATH = BERT_DATA_DIR / 'split_manifest.csv'
TRAINING_PAIRS_PATH = BERT_DATA_DIR / 'training_pairs.csv'
FAISS_MAP_PATH      = BERT_DATA_DIR / 'faiss_id_map.json'

JOB_SKILLS_PATH     = WORKSPACE_ROOT / 'dataset/jobs/job_skills.csv'
SKILLS_MAP_PATH     = WORKSPACE_ROOT / 'dataset/mappings/skills.csv'
INDUSTRIES_MAP_PATH = WORKSPACE_ROOT / 'dataset/mappings/industries.csv'

# ── Output paths ───────────────────────────────────────────────────────────────
BEST_MODEL_PATH  = RESULTS_DIR / 'lorentzfm_best_model.pt'
HISTORY_CSV_PATH = RESULTS_DIR / 'training_history.csv'
TRAIN_CFG_PATH   = RESULTS_DIR / 'training_config.json'
METRICS_PATH     = RESULTS_DIR / 'lorentzfm_metrics.json'
PREDS_PATH       = RESULTS_DIR / 'lorentzfm_predictions.csv'

# ── Hyperparameters ────────────────────────────────────────────────────────────
SEED         = 42
EMBED_DIM    = 32          # Lorentz spatial embedding dim; Lorentz dim = 33
CURVATURE    = -1.0        # Hyperbolic curvature
DROPOUT      = 0.1
LR           = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 256
NUM_EPOCHS   = 50
NUM_NEG      = 4           # negatives per positive pair
PATIENCE     = 10          # early stopping patience
GRAD_CLIP    = 1.0
TOP_K        = 10

# ── SBERT reference results (do not modify) ────────────────────────────────────
SBERT_REF = {'P@10': 0.8078, 'MRR': 0.8478, 'NDCG@10': 0.8857, 'Hit@10': 0.9680}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Seed   : {SEED}')

In [ ]:
# ── Reproducibility ─────────────────────────────────────────────────────────────
def set_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seeds(SEED)
print('Seeds set.')

---
## Section 3 — Feature Engineering

### Feature Design

| Side | Fields | Dim |
|------|--------|-----|
| **Resume** | domain vector (15) + skill_count (1) + seniority (1) + edu_score (1) + cert_count (1) | **19** |
| **Job** | domain vector (15) + skill code vector (35) + exp_level (1) + has_salary (1) + domain_count (1) | **53** |

**Relevance ground truth** (identical to SBERT/BERT):  
`relevant = 1  iff  dot(resume_domain_vec, job_domain_vec) > 0`

In [ ]:
# ── Constants ───────────────────────────────────────────────────────────────────
DOMAIN_COLUMNS = [
    'dom_Machine_Learning___AI', 'dom_Data_Science___Analytics',
    'dom_Data_Engineering',      'dom_Cloud___DevOps',
    'dom_Software_Engineering',  'dom_Cybersecurity',
    'dom_Business_Intelligence', 'dom_Project_Management',
    'dom_Business_Analysis',     'dom_Finance___FinTech',
    'dom_Healthcare___BioTech',  'dom_Blockchain___Web3',
    'dom_IoT___Embedded_Systems','dom_Product_Management',
    'dom_Sales___Marketing',
]

SENIORITY_MAP = {
    'Unknown': 0, 'Internship': 1, 'Entry level': 2, 'Associate': 3,
    'Mid-Senior level': 4, 'Director': 5, 'Executive': 6,
}
EXP_LEVEL_MAP = SENIORITY_MAP.copy()

RESUME_FEATURE_DIM = 19
JOB_FEATURE_DIM    = 53

# ── Resume fields (5 fields, total 19-dim) ──────────────────────────────────────
RESUME_FIELDS = [
    ('domain',    15),
    ('skill_cnt',  1),
    ('seniority',  1),
    ('edu_score',  1),
    ('cert_cnt',   1),
]

# ── Job fields (5 fields, total 53-dim) ────────────────────────────────────────
JOB_FIELDS = [
    ('domain',     15),
    ('skill_vec',  35),
    ('exp_level',   1),
    ('has_salary',  1),
    ('dom_count',   1),
]

print('Feature dims confirmed: Resume =', RESUME_FEATURE_DIM, '| Job =', JOB_FEATURE_DIM)

In [ ]:
def _norm_scalar(series: pd.Series, clip_max: float) -> pd.Series:
    """Normalise scalar feature: clip to [0, clip_max] then divide."""
    return series.fillna(0.0).clip(0, clip_max) / clip_max


def build_resume_feature_matrix(
    resumes_path: Path,
    resume_domain_path: Path,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns
    -------
    resume_ids  : (N,)    int64
    feature_mat : (N, 19) float32
    """
    resumes_df = pd.read_csv(resumes_path)
    dom_df     = pd.read_csv(resume_domain_path)
    df = resumes_df.merge(dom_df, on='resume_id', how='left')

    dom_mat    = df[DOMAIN_COLUMNS].fillna(0).values.astype(np.float32)
    skill_cnt  = _norm_scalar(df['skill_count'],   80.0).values.reshape(-1,1).astype(np.float32)
    seniority  = (df['seniority_level'].map(SENIORITY_MAP).fillna(0).values / 6.0
                  ).reshape(-1,1).astype(np.float32)
    edu_score  = _norm_scalar(df['education_score'], 4.0).values.reshape(-1,1).astype(np.float32)
    cert_cnt   = _norm_scalar(df['cert_count'],      10.0).values.reshape(-1,1).astype(np.float32)

    feature_mat = np.hstack([dom_mat, skill_cnt, seniority, edu_score, cert_cnt])
    resume_ids  = df['resume_id'].values.astype(np.int64)
    print(f'Resume feature matrix: {feature_mat.shape}')
    return resume_ids, feature_mat


def build_job_feature_matrix(
    domain_labels_path: Path,
    split_manifest_path: Path,
    job_skills_path: Path,
    skills_mapping_path: Path,
    faiss_id_map_path: Path,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns
    -------
    job_ids     : (M,)    int64  (FAISS index order)
    feature_mat : (M, 53) float32
    """
    dom_df      = pd.read_csv(domain_labels_path)
    manifest_df = pd.read_csv(split_manifest_path,
                              usecols=['job_id','exp_level_label','has_salary','domain_count'])

    skills_map_df  = pd.read_csv(skills_mapping_path)
    skill_codes    = sorted(skills_map_df['skill_abr'].tolist())
    skill_code_idx = {c: i for i, c in enumerate(skill_codes)}
    num_skills     = len(skill_codes)   # 35

    job_skills_df = pd.read_csv(job_skills_path)
    skill_records = job_skills_df[job_skills_df['skill_abr'].isin(skill_code_idx)].copy()
    skill_records['skill_idx'] = skill_records['skill_abr'].map(skill_code_idx)
    skill_pivot = (
        skill_records.groupby('job_id')['skill_idx']
        .apply(list).reset_index(name='skill_indices')
    )

    with open(faiss_id_map_path, 'r') as f:
        faiss_map = json.load(f)
    ordered_job_ids = np.array(
        [faiss_map[str(i)] for i in range(len(faiss_map))], dtype=np.int64
    )
    M = len(ordered_job_ids)

    base_df = pd.DataFrame({'job_id': ordered_job_ids})
    base_df = base_df.merge(
        dom_df[['job_id'] + DOMAIN_COLUMNS + ['domain_count','has_domain']],
        on='job_id', how='left'
    )
    base_df = base_df.merge(manifest_df, on='job_id', how='left')
    base_df = base_df.merge(skill_pivot,  on='job_id', how='left')

    dom_mat   = base_df[DOMAIN_COLUMNS].fillna(0).values.astype(np.float32)
    skill_mat = np.zeros((M, num_skills), dtype=np.float32)
    for row_idx, indices in enumerate(base_df['skill_indices'].values):
        if isinstance(indices, list):
            for si in indices:
                skill_mat[row_idx, si] = 1.0

    exp_col = (
        base_df['exp_level_label'].map(EXP_LEVEL_MAP).fillna(0).values / 6.0
    ).reshape(-1,1).astype(np.float32)
    has_salary = base_df['has_salary'].fillna(0).values.reshape(-1,1).astype(np.float32)
    dom_count_col = (
        base_df['domain_count_x'] if 'domain_count_x' in base_df.columns
        else base_df['domain_count']
    )
    dom_count = _norm_scalar(dom_count_col, 5.0).values.reshape(-1,1).astype(np.float32)

    feature_mat = np.hstack([dom_mat, skill_mat, exp_col, has_salary, dom_count])
    assert feature_mat.shape == (M, JOB_FEATURE_DIM), f'Shape mismatch: {feature_mat.shape}'
    print(f'Job feature matrix   : {feature_mat.shape}')
    return ordered_job_ids, feature_mat


def compute_relevance_matrix(
    resume_domain_mat: np.ndarray,  # (N, 15)
    job_domain_mat:    np.ndarray,  # (M, 15)
) -> np.ndarray:
    """Returns (N, M) bool — relevant iff dot(resume_dom, job_dom) > 0."""
    dot = resume_domain_mat @ job_domain_mat.T
    return (dot > 0).astype(np.bool_)

In [ ]:
# ── Build feature matrices ──────────────────────────────────────────────────────
print('Loading resume features...')
resume_ids, resume_feat_mat = build_resume_feature_matrix(
    RESUMES_PATH, RESUME_DOMAIN_PATH
)

print('\nLoading job features (may take ~3–5 s)...')
t0 = time.time()
job_ids, job_feat_mat = build_job_feature_matrix(
    DOMAIN_LABELS_PATH, SPLIT_MANIFEST_PATH,
    JOB_SKILLS_PATH, SKILLS_MAP_PATH, FAISS_MAP_PATH
)
print(f'  Loaded in {time.time()-t0:.2f} s')

# Build index maps
resume_id_to_idx = {int(rid): i for i, rid in enumerate(resume_ids)}
job_id_to_idx    = {int(jid): i for i, jid in enumerate(job_ids)}

resume_domain_mat = resume_feat_mat[:, :15]
job_domain_mat    = job_feat_mat[:, :15]

print(f'\nResumes          : {len(resume_ids)}')
print(f'Jobs             : {len(job_ids)}')
print(f'Resume feat dim  : {resume_feat_mat.shape[1]}')
print(f'Job feat dim     : {job_feat_mat.shape[1]}')

In [ ]:
# ── Split indices ───────────────────────────────────────────────────────────────
manifest_df = pd.read_csv(SPLIT_MANIFEST_PATH, usecols=['job_id','split'])
splits = manifest_df['split'].value_counts().to_dict()
print('Split distribution:', splits)

train_job_ids = set(manifest_df[manifest_df['split'] == 'train']['job_id'].tolist())
val_job_ids   = set(manifest_df[manifest_df['split'] == 'val'  ]['job_id'].tolist())
test_job_ids  = set(manifest_df[manifest_df['split'] == 'test' ]['job_id'].tolist())

train_job_indices = np.array(
    [job_id_to_idx[j] for j in train_job_ids if j in job_id_to_idx], dtype=np.int64
)
val_job_indices = np.array(
    [job_id_to_idx[j] for j in val_job_ids if j in job_id_to_idx], dtype=np.int64
)

print(f'\nTrain job indices : {len(train_job_indices)}')
print(f'Val   job indices : {len(val_job_indices)}')
print(f'Test  job count   : {len(test_job_ids)}  (NEVER used during training)')

---
## Section 4 — LorentzFM Model

### Architecture

```
Resume Features (19-dim)                   Job Features (53-dim)
      ↓                                           ↓
Split → 5 fields                           Split → 5 fields
      ↓                                           ↓
Linear → LayerNorm → ×0.1              Linear → LayerNorm → ×0.1
      ↓                                           ↓
exp_map_origin → Lorentz point         exp_map_origin → Lorentz point
      ↓_______________________________________↓
         score = Σ_{i,j} −⟨e_i^R, e_j^J⟩_L + bias
```

**Lorentz inner product:** `⟨x,y⟩_L = −x₀y₀ + Σ xᵢyᵢ`  
**exp_map_origin:** `x = (√(1+‖v‖²), v)` — maps any vector onto the unit hyperboloid  
**Loss:** BPR — `−log σ(score_pos − score_neg) + λ‖W‖²`

In [ ]:
# ── Lorentz Manifold Operations ─────────────────────────────────────────────────

def lorentz_inner(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Minkowski inner product: <x,y>_L = -x0*y0 + sum(xi*yi)"""
    x0, xi = x[..., :1], x[..., 1:]
    y0, yi = y[..., :1], y[..., 1:]
    return (-x0*y0 + (xi*yi).sum(dim=-1, keepdim=True)).squeeze(-1)


def exp_map_origin(v: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Map spatial vector v onto the Lorentz hyperboloid: (sqrt(1+||v||^2), v)"""
    v_norm_sq = (v*v).sum(dim=-1, keepdim=True).clamp(min=0.0)
    x0 = torch.sqrt(1.0 + v_norm_sq + eps)
    return torch.cat([x0, v], dim=-1)


def lorentz_norm(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Lorentz norm — should equal 1 for unit hyperboloid."""
    return torch.sqrt((-lorentz_inner(x, x)).clamp(min=eps))


# Quick sanity check
v   = torch.randn(4, 8)
pts = exp_map_origin(v)
constraint = -pts[:, 0]**2 + (pts[:, 1:]**2).sum(1)
print(f'Hyperboloid constraint (should be ≈ -1): mean={constraint.mean():.6f}')
print(f'Lorentz norms (should be ≈ 1.0):         {lorentz_norm(pts).mean():.6f}')

In [ ]:
# ── FeatureFieldEncoder ─────────────────────────────────────────────────────────

class FeatureFieldEncoder(nn.Module):
    """
    Encodes one feature field (input_dim) → Lorentz point (embed_dim+1).
    Pipeline: Linear → LayerNorm → scale×0.1 → exp_map_origin
    """
    def __init__(self, input_dim: int, embed_dim: int, field_name: str = ''):
        super().__init__()
        self.field_name = field_name
        self.proj = nn.Linear(input_dim, embed_dim, bias=True)
        self.ln   = nn.LayerNorm(embed_dim)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spatial = self.ln(self.proj(x)) * 0.1
        return exp_map_origin(spatial)   # (..., embed_dim + 1)

In [ ]:
# ── LorentzFM Model ─────────────────────────────────────────────────────────────

class LorentzFM(nn.Module):
    """
    LorentzFM Baseline for Resume-to-Job Recommendation.

    score(R,J) = Σ_{i=1..5} Σ_{j=1..5}  −⟨e_i^R, e_j^J⟩_L  + bias
    """
    def __init__(self, embed_dim: int = 32, curvature: float = -1.0,
                 dropout_prob: float = 0.1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.curvature   = curvature
        self.lorentz_dim = embed_dim + 1

        self.resume_encoders = nn.ModuleList([
            FeatureFieldEncoder(in_d, embed_dim, name)
            for name, in_d in RESUME_FIELDS
        ])
        self.job_encoders = nn.ModuleList([
            FeatureFieldEncoder(in_d, embed_dim, name)
            for name, in_d in JOB_FIELDS
        ])
        self.dropout = nn.Dropout(p=dropout_prob)
        self.bias    = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _slice_resume(r):
        return [r[:,:15], r[:,15:16], r[:,16:17], r[:,17:18], r[:,18:19]]

    @staticmethod
    def _slice_job(j):
        return [j[:,:15], j[:,15:50], j[:,50:51], j[:,51:52], j[:,52:53]]

    def encode_resume(self, r: torch.Tensor) -> torch.Tensor:
        """(B,19) → (B,5,lorentz_dim)"""
        return torch.cat([
            enc(self.dropout(f)).unsqueeze(1)
            for enc, f in zip(self.resume_encoders, self._slice_resume(r))
        ], dim=1)

    def encode_job(self, j: torch.Tensor) -> torch.Tensor:
        """(B,53) → (B,5,lorentz_dim)"""
        return torch.cat([
            enc(self.dropout(f)).unsqueeze(1)
            for enc, f in zip(self.job_encoders, self._slice_job(j))
        ], dim=1)

    def interaction_score(
        self,
        r_lorentz: torch.Tensor,   # (B, 5, D)
        j_lorentz: torch.Tensor,   # (B, 5, D)
    ) -> torch.Tensor:             # (B,)
        r_exp  = r_lorentz.unsqueeze(2)          # (B,5,1,D)
        j_exp  = j_lorentz.unsqueeze(1)          # (B,1,5,D)
        r_t, r_s = r_exp[...,:1], r_exp[...,1:]
        j_t, j_s = j_exp[...,:1], j_exp[...,1:]
        linner = -r_t*j_t + (r_s*j_s).sum(dim=-1, keepdim=True)  # (B,5,5,1)
        return (-linner.squeeze(-1)).sum(dim=[1,2]) + self.bias    # (B,)

    def forward(self, r: torch.Tensor, j: torch.Tensor) -> torch.Tensor:
        """Returns (B,) compatibility scores."""
        return self.interaction_score(self.encode_resume(r), self.encode_job(j))

    @torch.no_grad()
    def score_resume_vs_all_jobs(
        self,
        r_feat:    torch.Tensor,   # (19,) or (1,19)
        j_feat_mat: torch.Tensor,  # (M,53)
        batch_size: int = 2048,
        device:     Optional[torch.device] = None,
    ) -> np.ndarray:
        """Scores one resume against all M jobs. Returns (M,) float32."""
        self.eval()
        if r_feat.dim() == 1:
            r_feat = r_feat.unsqueeze(0)
        if device is None:
            device = next(self.parameters()).device
        r_feat = r_feat.to(device)
        r_lor  = self.encode_resume(r_feat)   # (1,5,D)
        M = j_feat_mat.shape[0]
        all_scores = np.empty(M, dtype=np.float32)
        for s in range(0, M, batch_size):
            e   = min(s + batch_size, M)
            jf  = j_feat_mat[s:e].to(device)
            j_l = self.encode_job(jf)
            r_e = r_lor.expand(e-s, -1, -1)
            all_scores[s:e] = self.interaction_score(r_e, j_l).cpu().numpy()
        return all_scores


# ── BPR Loss ────────────────────────────────────────────────────────────────────

class BPRLoss(nn.Module):
    """BPR: L = -log σ(score_pos - score_neg) + λ * Σ||W_proj||²"""
    def __init__(self, l2_lambda: float = 1e-5):
        super().__init__()
        self.l2_lambda = l2_lambda

    def forward(self, s_pos, s_neg, model=None):
        bpr = -F.logsigmoid(s_pos - s_neg).mean()
        if model is not None and self.l2_lambda > 0:
            l2 = sum(
                p.pow(2).sum()
                for n, p in model.named_parameters()
                if 'proj.weight' in n
            )
            bpr = bpr + self.l2_lambda * l2
        return bpr


# ── Instantiate & verify ────────────────────────────────────────────────────────
model   = LorentzFM(embed_dim=EMBED_DIM, curvature=CURVATURE, dropout_prob=DROPOUT)
loss_fn = BPRLoss(l2_lambda=WEIGHT_DECAY)

total_params = sum(p.numel() for p in model.parameters())
print(f'LorentzFM instantiated')
print(f'  embed_dim    : {EMBED_DIM}')
print(f'  lorentz_dim  : {EMBED_DIM+1}')
print(f'  curvature    : {CURVATURE}')
print(f'  total params : {total_params:,}')

# Quick forward pass check
r_dummy = torch.randn(8, RESUME_FEATURE_DIM)
j_dummy = torch.randn(8, JOB_FEATURE_DIM)
scores  = model(r_dummy, j_dummy)
print(f'  forward pass : input {r_dummy.shape}, {j_dummy.shape} → scores {scores.shape} ✓')
assert not torch.isnan(scores).any(), 'NaN in forward pass!'

---
## Section 5 — Training Dataset & DataLoader

In [ ]:
class LorentzFMPairDataset(Dataset):
    """
    Pairwise BPR dataset.
    Each item: (resume_feat, pos_job_feat, neg_job_feat)
    Negatives sampled from train-split only (never val/test).
    """
    def __init__(
        self,
        training_pairs_path: Path,
        resume_ids:   np.ndarray,
        resume_feat:  np.ndarray,
        job_ids:      np.ndarray,
        job_feat:     np.ndarray,
        split_manifest_path: Path,
        num_negatives: int = 4,
        seed: int = 42,
    ):
        super().__init__()
        self.num_negatives = num_negatives
        self.rng = np.random.default_rng(seed)

        self.r_idx_map = {int(rid): i for i, rid in enumerate(resume_ids)}
        self.j_idx_map = {int(jid): i for i, jid in enumerate(job_ids)}
        self.resume_feat = resume_feat
        self.job_feat    = job_feat

        pairs = pd.read_csv(training_pairs_path)
        pairs = pairs[pairs['label'] == 1]
        pairs = pairs[
            pairs['resume_id'].isin(self.r_idx_map) &
            pairs['job_id'].isin(self.j_idx_map)
        ].reset_index(drop=True)
        print(f'  Positive training pairs loaded: {len(pairs):,}')

        self.r_indices   = pairs['resume_id'].map(self.r_idx_map).values
        self.pos_indices = pairs['job_id'].map(self.j_idx_map).values

        # Negative pool — train-split only
        mf = pd.read_csv(split_manifest_path, usecols=['job_id','split'])
        train_jids = set(mf[mf['split'] == 'train']['job_id'].tolist())
        self.neg_pool = np.array(
            [self.j_idx_map[j] for j in train_jids if j in self.j_idx_map], dtype=np.int64
        )
        print(f'  Negative pool (train-split jobs): {len(self.neg_pool):,}')

        self.r_dom = resume_feat[:, :15]
        self.j_dom = job_feat[:, :15]

    def __len__(self):
        return len(self.r_indices) * self.num_negatives

    def __getitem__(self, idx):
        pair_idx = idx // self.num_negatives
        r_idx = int(self.r_indices[pair_idx])
        p_idx = int(self.pos_indices[pair_idx])
        r_dom = self.r_dom[r_idx]
        for _ in range(20):
            n_idx = int(self.rng.choice(self.neg_pool))
            if np.dot(r_dom, self.j_dom[n_idx]) == 0:
                break
        return {
            'resume_feat':  torch.from_numpy(self.resume_feat[r_idx].copy()),
            'pos_job_feat': torch.from_numpy(self.job_feat[p_idx].copy()),
            'neg_job_feat': torch.from_numpy(self.job_feat[n_idx].copy()),
        }


print('Building training dataset...')
train_dataset = LorentzFMPairDataset(
    training_pairs_path = TRAINING_PAIRS_PATH,
    resume_ids          = resume_ids,
    resume_feat         = resume_feat_mat,
    job_ids             = job_ids,
    job_feat            = job_feat_mat,
    split_manifest_path = SPLIT_MANIFEST_PATH,
    num_negatives       = NUM_NEG,
    seed                = SEED,
)
print(f'  Total training samples : {len(train_dataset):,}')

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 0,
    drop_last   = False,
)
print(f'  Batches per epoch      : {len(train_loader)}')

In [ ]:
# ── Validation: P@10 on val job pool ────────────────────────────────────────────
print('Building validation evaluation set...')

val_job_feat   = job_feat_mat[val_job_indices]          # (7491, 53)
val_relevance  = compute_relevance_matrix(
    resume_domain_mat,
    job_feat_mat[val_job_indices, :15],
)   # (228, 7491)

val_r_feat_t = torch.from_numpy(resume_feat_mat)        # (228, 19)
val_j_feat_t = torch.from_numpy(val_job_feat)           # (7491, 53)

print(f'  Val job pool size       : {len(val_job_indices)}')
print(f'  Val relevance matrix    : {val_relevance.shape}')
print(f'  Val positive rate       : {100*val_relevance.mean():.1f}%')


def compute_val_p10(mdl: LorentzFM) -> float:
    """Mean P@10 over all 228 resumes on val pool. Negated for early stopping."""
    mdl.eval()
    precisions = []
    with torch.no_grad():
        for r_i in range(len(resume_feat_mat)):
            r_f    = val_r_feat_t[r_i].unsqueeze(0)
            scores = mdl.score_resume_vs_all_jobs(r_f, val_j_feat_t, batch_size=1024,
                                                   device=DEVICE)
            top_idx = np.argsort(-scores)[:TOP_K]
            precisions.append(val_relevance[r_i, top_idx].sum() / TOP_K)
    return float(np.mean(precisions))

---
## Section 6 — Training Loop with Early Stopping

In [ ]:
# ── Re-initialise model & optimiser ─────────────────────────────────────────────
set_seeds(SEED)

model   = LorentzFM(embed_dim=EMBED_DIM, curvature=CURVATURE, dropout_prob=DROPOUT).to(DEVICE)
loss_fn = BPRLoss(l2_lambda=WEIGHT_DECAY)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
)

history        = []
best_val_score = -float('inf')   # best (positive) P@10
best_epoch     = 0
patience_count = 0
train_start    = time.time()

print(f'Starting training — {NUM_EPOCHS} epochs max, patience={PATIENCE}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')
print('─' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss, n_batches = 0.0, 0
    epoch_start = time.time()

    for batch in train_loader:
        r_f  = batch['resume_feat'].to(DEVICE)
        p_f  = batch['pos_job_feat'].to(DEVICE)
        n_f  = batch['neg_job_feat'].to(DEVICE)

        s_pos = model(r_f, p_f)
        s_neg = model(r_f, n_f)
        loss  = loss_fn(s_pos, s_neg, model=model)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

    avg_train_loss = epoch_loss / max(n_batches, 1)

    # ── Validate ───────────────────────────────────────────────────────────────
    val_p10 = compute_val_p10(model)
    scheduler.step(-val_p10)   # scheduler minimises; negate P@10

    current_lr    = optimizer.param_groups[0]['lr']
    epoch_elapsed = time.time() - epoch_start
    total_elapsed = time.time() - train_start

    print(
        f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
        f'train_loss={avg_train_loss:.6f} | '
        f'val_P@10={val_p10:.4f} | '
        f'lr={current_lr:.6f} | '
        f'{epoch_elapsed:.1f}s'
    )

    history.append({
        'epoch':          epoch,
        'train_loss':     round(avg_train_loss, 6),
        'val_P@10':       round(val_p10, 4),
        'learning_rate':  current_lr,
        'epoch_time_s':   round(epoch_elapsed, 2),
        'elapsed_total_s': round(total_elapsed, 2),
    })

    # ── Model selection ────────────────────────────────────────────────────────
    if val_p10 > best_val_score:
        best_val_score = val_p10
        best_epoch     = epoch
        patience_count = 0
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_p10':     val_p10,
            'train_loss':  avg_train_loss,
        }, BEST_MODEL_PATH)
        print(f'  ✓ Best model saved  (val_P@10={val_p10:.4f}, epoch={epoch})')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}. Best epoch={best_epoch}')
            break

total_time = time.time() - train_start
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV_PATH, index=False)

print('─' * 70)
print(f'Training complete!')
print(f'  Best epoch          : {best_epoch} / {len(history)}')
print(f'  Best val P@10       : {best_val_score:.4f}')
print(f'  Total training time : {total_time:.1f} s ({total_time/60:.2f} min)')
print(f'  Model saved to      : {BEST_MODEL_PATH}')

---
## Section 7 — Training Curves

In [ ]:
hist = pd.read_csv(HISTORY_CSV_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(hist['epoch'], hist['train_loss'], color='steelblue', linewidth=2, label='Train Loss')
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BPR Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation P@10
axes[1].plot(hist['epoch'], hist['val_P@10'], color='darkorange', linewidth=2, label='Val P@10')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
axes[1].scatter([best_epoch], [best_val_score], color='red', zorder=5, s=80,
                label=f'Best P@10 = {best_val_score:.4f}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Precision@10', fontsize=12)
axes[1].set_title('Validation P@10', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LorentzFM Training Curves — CareerCompass AI Baseline',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# ── Training history table (last 10 epochs) ─────────────────────────────────────
print(hist.tail(10).to_string(index=False))

---
## Section 8 — Evaluation on Full Candidate Pool

**Metrics** (identical to SBERT/BERT baselines):
- **Precision@10** — fraction of top-10 results that are relevant
- **MRR** — reciprocal rank of the first relevant result
- **NDCG@10** — discounted cumulative gain at position 10
- **Hit@10** — 1 if any relevant result appears in top-10

**Candidate pool:** all 123,849 jobs (identical to SBERT/BERT)  
**Ground truth:** `dot(resume_domain_vector, job_domain_vector) > 0`

In [ ]:
# ── Ranking metric functions ─────────────────────────────────────────────────────

def precision_at_k(rel: np.ndarray, k: int = 10) -> float:
    return float(rel[:k].sum()) / k


def mean_reciprocal_rank(rel: np.ndarray) -> float:
    hits = np.where(rel)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)


def ndcg_at_k(rel: np.ndarray, k: int = 10) -> float:
    top_k = rel[:k]
    dcg   = sum(r / math.log2(i + 2) for i, r in enumerate(top_k))
    ideal = min(int(rel.sum()), k)
    idcg  = sum(1.0 / math.log2(i + 2) for i in range(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def hit_at_k(rel: np.ndarray, k: int = 10) -> float:
    return float(rel[:k].sum() > 0)


print('Metric functions defined.')

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────────
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model = model.to(DEVICE)
model.eval()

print(f'Loaded best checkpoint: epoch {checkpoint["epoch"]}  '
      f'val_P@10={checkpoint["val_p10"]:.4f}')

In [ ]:
# ── Compute ground-truth relevance for all 228 × 123,849 pairs ──────────────────
print('Computing relevance matrix (228 × 123,849)...')
t0 = time.time()
relevance_mat = compute_relevance_matrix(
    resume_feat_mat[:, :15],
    job_feat_mat[:, :15],
)
print(f'  Shape: {relevance_mat.shape}  '
      f'positive rate: {100*relevance_mat.mean():.2f}%  '
      f'({time.time()-t0:.2f} s)')

In [ ]:
# ── Per-resume evaluation ────────────────────────────────────────────────────────
print(f'Evaluating {len(resume_ids)} resumes × {len(job_ids):,} jobs...')
job_feat_tensor = torch.from_numpy(job_feat_mat)

per_resume  = []
pred_rows   = []
t_eval      = time.time()

for r_idx, r_id in enumerate(resume_ids):
    r_feat = torch.from_numpy(resume_feat_mat[r_idx])

    scores  = model.score_resume_vs_all_jobs(
        r_feat, job_feat_tensor, batch_size=2048, device=DEVICE
    )
    ranked  = np.argsort(-scores)
    rel_ord = relevance_mat[r_idx, ranked].astype(np.float32)

    p10  = precision_at_k(rel_ord)
    mrr  = mean_reciprocal_rank(rel_ord)
    ndcg = ndcg_at_k(rel_ord)
    hit  = hit_at_k(rel_ord)

    per_resume.append({
        'resume_id': int(r_id),
        'P@10': round(p10, 4), 'MRR': round(mrr, 4),
        'NDCG@10': round(ndcg, 4), 'Hit@10': round(hit, 4),
        'num_relevant': int(rel_ord.sum()),
    })

    for rank, j_idx in enumerate(ranked[:TOP_K], 1):
        pred_rows.append({
            'resume_id': int(r_id), 'job_id': int(job_ids[j_idx]),
            'score': round(float(scores[j_idx]), 6),
            'rank': rank, 'relevant': int(relevance_mat[r_idx, j_idx]),
        })

    if (r_idx + 1) % 50 == 0 or r_idx == 0:
        print(f'  [{r_idx+1:3d}/228]  elapsed: {time.time()-t_eval:.1f}s')

total_eval_time = time.time() - t_eval
print(f'\nEvaluation complete: {total_eval_time:.2f}s  ({total_eval_time/len(resume_ids):.2f}s/resume)')

In [ ]:
# ── Aggregate & display results ──────────────────────────────────────────────────
metrics_df = pd.DataFrame(per_resume)

mean_p10  = metrics_df['P@10'].mean()
mean_mrr  = metrics_df['MRR'].mean()
mean_ndcg = metrics_df['NDCG@10'].mean()
mean_hit  = metrics_df['Hit@10'].mean()

print('=' * 55)
print('  LorentzFM Results  (Full 123,849-job Candidate Pool)')
print('=' * 55)
print(f'  P@10    = {mean_p10:.4f}')
print(f'  MRR     = {mean_mrr:.4f}')
print(f'  NDCG@10 = {mean_ndcg:.4f}')
print(f'  Hit@10  = {mean_hit:.4f}')
print('=' * 55)

# Save outputs
pd.DataFrame(pred_rows).to_csv(PREDS_PATH, index=False)
metrics_df.to_csv(RESULTS_DIR / 'lorentzfm_per_resume_metrics.csv', index=False)
print(f'\nPredictions saved : {PREDS_PATH}')

In [ ]:
# ── Save metrics JSON ────────────────────────────────────────────────────────────
metrics_record = {
    'model': 'LorentzFM',
    'best_epoch': int(checkpoint['epoch']),
    'val_P@10_at_best_epoch': float(checkpoint['val_p10']),
    'evaluation': {
        'candidate_pool_size': int(len(job_ids)),
        'num_resumes':         int(len(resume_ids)),
        'top_k': TOP_K,
        'ground_truth': 'dot(resume_domain_vector, job_domain_vector) > 0',
        'evaluation_time_s': round(total_eval_time, 2),
    },
    'results': {
        'P@10':    round(mean_p10,  4),
        'MRR':     round(mean_mrr,  4),
        'NDCG@10': round(mean_ndcg, 4),
        'Hit@10':  round(mean_hit,  4),
    },
    'sbert_reference': SBERT_REF,
    'delta_vs_sbert': {
        'P@10':    round(mean_p10  - SBERT_REF['P@10'],    4),
        'MRR':     round(mean_mrr  - SBERT_REF['MRR'],     4),
        'NDCG@10': round(mean_ndcg - SBERT_REF['NDCG@10'], 4),
        'Hit@10':  round(mean_hit  - SBERT_REF['Hit@10'],  4),
    },
    'test_split_used_during_training': False,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics_record, f, indent=2)
print('Metrics saved:', METRICS_PATH)

---
## Section 9 — Results & Comparison with SBERT

In [ ]:
# ── Comparison table ─────────────────────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Model': 'SBERT',     **SBERT_REF},
    {'Model': 'LorentzFM', 'P@10': round(mean_p10,4),
                            'MRR': round(mean_mrr,4),
                            'NDCG@10': round(mean_ndcg,4),
                            'Hit@10': round(mean_hit,4)},
])
print('\nFinal Comparison')
print(comparison.to_string(index=False))

print('\nΔ LorentzFM − SBERT')
for m in ['P@10','MRR','NDCG@10','Hit@10']:
    diff = comparison.loc[comparison.Model=='LorentzFM',m].values[0] - \
           comparison.loc[comparison.Model=='SBERT', m].values[0]
    print(f'  {m:<10} {diff:+.4f}')

In [ ]:
# ── Comparison bar chart ─────────────────────────────────────────────────────────
metrics_list = ['P@10', 'MRR', 'NDCG@10', 'Hit@10']
models_list  = ['SBERT', 'LorentzFM']
colors       = ['#1f77b4', '#2ca02c']

x     = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
for i, (mdl, clr) in enumerate(zip(models_list, colors)):
    vals = [comparison.loc[comparison.Model==mdl, m].values[0] for m in metrics_list]
    bars = ax.bar(x + (i - 0.5) * width, vals, width, label=mdl, color=clr)
    ax.bar_label(bars, fmt='%.4f', fontsize=9, padding=3)

ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_title('CareerCompass AI — LorentzFM vs SBERT Ranking Metrics',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison_plot.png', dpi=150)
plt.show()
print('Comparison plot saved.')

In [ ]:
# ── Per-resume metric distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metric_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

for ax, metric, clr in zip(axes, ['P@10','MRR','NDCG@10','Hit@10'], metric_colors):
    ax.hist(metrics_df[metric], bins=20, color=clr, edgecolor='white', alpha=0.85)
    ax.axvline(metrics_df[metric].mean(), color='black', linestyle='--',
               linewidth=1.5, label=f'mean={metrics_df[metric].mean():.4f}')
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('LorentzFM — Per-Resume Metric Distribution (228 Resumes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_resume_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Sample top-10 predictions for first 3 resumes ───────────────────────────────
preds_df = pd.read_csv(PREDS_PATH)
print('Sample top-10 predictions (first 3 resumes):')
print(preds_df.head(30).to_string(index=False))

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────────
print('=' * 55)
print('  LORENTZFM EXPERIMENT SUMMARY')
print('=' * 55)
print(f'  Model         : LorentzFM')
print(f'  Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Best Epoch    : {checkpoint["epoch"]}')
print(f'  Val P@10      : {checkpoint["val_p10"]:.4f}')
print(f'  Training time : {total_time:.1f}s ({total_time/60:.2f} min)')
print(f'  Eval time     : {total_eval_time:.1f}s')
print('─' * 55)
print(f'  Metric   LorentzFM    SBERT     Delta')
print('─' * 55)
for m in ['P@10','MRR','NDCG@10','Hit@10']:
    lv = metrics_record['results'][m]
    sv = SBERT_REF[m]
    print(f'  {m:<10} {lv:.4f}      {sv:.4f}   {lv-sv:+.4f}')
print('=' * 55)
print('  Test split was NEVER used during training.')
print('  All metrics are actual experimental results.')
print('=' * 55)